In [43]:
#imports 
import pandas as pd
import numpy as np
from pathlib import Path

RAW_PATH = Path("../data/raw/indian_roads_dataset.csv")
OUT_PATH = Path("../data/processed/accidents_clean.csv")

df = pd.read_csv(RAW_PATH)
print("Raw shape:", df.shape)
df.head()

Raw shape: (20000, 24)


,accident_id,city,state,latitude,longitude,date,time,hour,day_of_week,is_weekend,...,visibility,temperature,traffic_density,cause,accident_severity,vehicles_involved,casualties,is_peak_hour,festival,risk_score
0,0,Pune,Maharashtra,18.680827,73.930388,2023-10-22,5:00,5,Sunday,1,...,low,32,high,weather,fatal,2,2,0,NaN,0.85
1,1,Mumbai,Maharashtra,18.817732,72.790846,2023-05-21,4:00,4,Sunday,1,...,high,34,low,weather,major,4,3,0,NaN,0.10
2,2,Mumbai,Maharashtra,19.096889,72.819424,2024-07-10,13:00,13,Wednesday,0,...,low,21,medium,weather,minor,1,1,0,NaN,0.45
3,3,Chandigarh,Punjab,30.787805,76.847507,2025-03-30,11:00,11,Sunday,1,...,low,30,high,distraction,minor,5,2,0,NaN,0.65
4,4,Chennai,Tamil Nadu,12.965155,80.283313,2024-01-25,16:00,16,Thursday,0,...,high,24,low,distraction,minor,2,1,0,NaN,0.10


In [44]:
# Clean column names
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)

In [45]:
# Convert date and time columns to datetime format
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["time"] = pd.to_datetime(df["time"], format="%H:%M", errors="coerce").dt.time
df["datetime"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["time"].astype(str),
    errors="coerce"
)
df[["date", "time", "datetime"]].head()

,date,time,datetime
0,2023-10-22,05:00:00,2023-10-22 05:00:00
1,2023-05-21,04:00:00,2023-05-21 04:00:00
2,2024-07-10,13:00:00,2024-07-10 13:00:00
3,2025-03-30,11:00:00,2025-03-30 11:00:00
4,2024-01-25,16:00:00,2024-01-25 16:00:00


In [46]:
# Convert boolean columns to boolean type
bool_cols = ["is_weekend", "traffic_signal", "is_peak_hour"]
for col in bool_cols:
    df[col] = df[col].astype(bool)
print(df[bool_cols].dtypes)

is_weekend        bool
traffic_signal    bool
is_peak_hour      bool
dtype: object


In [47]:
# Fill missing values in the "festival" column with "None"
df["festival"] = df["festival"].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
df["festival"] = df["festival"].fillna("None")
print(df["festival"].value_counts())

festival
None        19885
Holi           38
Eid            34
Diwali         31
New Year       12
Name: count, dtype: int64


In [48]:
# Convert numeric columns to numeric type
numeric_cols = [
    "latitude", "longitude", "hour", "lanes",
    "temperature", "vehicles_involved", "casualties", "risk_score"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols].dtypes

latitude             float64
longitude            float64
hour                   int64
lanes                  int64
temperature            int64
vehicles_involved      int64
casualties             int64
risk_score           float64
dtype: object

In [49]:
# Standardize text values to title case
title_cols = [
    "city", "state", "day_of_week",
    "road_type", "weather", "traffic_density",
    "cause", "accident_severity", "visibility"
]
for col in title_cols:
    df[col] = df[col].str.title()

for col in title_cols:
    print(f"{col}: {df[col].unique()[:8]}")

city: ['Pune' 'Mumbai' 'Chandigarh' 'Chennai' 'Delhi' 'Bangalore' 'Hyderabad'
 'Kolkata']
state: ['Maharashtra' 'Punjab' 'Tamil Nadu' 'Delhi' 'Karnataka' 'Telangana'
 'West Bengal']
day_of_week: ['Sunday' 'Wednesday' 'Thursday' 'Monday' 'Friday' 'Tuesday' 'Saturday']
road_type: ['Highway' 'Urban' 'Rural']
weather: ['Fog' 'Clear' 'Rain']
traffic_density: ['High' 'Low' 'Medium']
cause: ['Weather' 'Distraction' 'Overspeeding' 'Drunk Driving' 'Poor Road']
accident_severity: ['Fatal' 'Major' 'Minor']
visibility: ['Low' 'High' 'Medium']


In [50]:
# Feature Engineering: Create new columns for year, month, day, quarter, and week of the year as well as boolean columns for fatal accidents and festivals
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.strftime("%b")
df["day"] = df["date"].dt.day
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype("Int64")

df["is_fatal"] = df["accident_severity"].str.lower().eq("fatal")
df["is_festival"] = df["festival"].ne("None")

In [51]:
# Validate 
print("Clean shape:", df.shape)
print("Total nulls:", df.isnull().sum().sum())
print("\nDtypes:\n", df.dtypes)
print("\nSeverity counts:\n", df["accident_severity"].value_counts())
print("\nFestival counts:\n", df["festival"].value_counts())
print("\nBool flags:")
print(df[["is_weekend", "traffic_signal", "is_peak_hour", "is_fatal", "is_festival"]].sum())

Clean shape: (20000, 33)
Total nulls: 0

Dtypes:
 accident_id                   int64
city                         object
state                        object
latitude                    float64
longitude                   float64
date                 datetime64[ns]
time                         object
hour                          int64
day_of_week                  object
is_weekend                     bool
road_type                    object
lanes                         int64
traffic_signal                 bool
weather                      object
visibility                   object
temperature                   int64
traffic_density              object
cause                        object
accident_severity            object
vehicles_involved             int64
casualties                    int64
is_peak_hour                   bool
festival                     object
risk_score                  float64
datetime             datetime64[ns]
year                          int32
month         

In [52]:
# Save the cleaned dataset to a new CSV file
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f"Saved → {OUT_PATH.resolve()}")

Saved → C:\Users\Himanshu\Documents\indian-road-accident-analysis\data\processed\accidents_clean.csv
